This .ipynb file provides two methods of comparing experiments. The first code block compares two LLM-inference experiments that use different models (e.g., Sonnet 4.6, Haiku 4.5). This file assumes that nothing else was changed between the experiments, except for the frontier model used.

The second code block compares an LLM-inference experiment to the latest supervised_ml/raw/ experiment. This provides a comparison between how well the model does compared to a supervised machine learning model using the same data. The third code block is a direct copy of the second, but changes the LLM-inference experiment to the second compared model. This allows you to compare, for example, Sonnet 4.6 to supervised ML, as well as Haiku 4.5 to supervised ML, and see the differences in the two by looking at the output differences of the two code blocks.

In [26]:
import json
import numpy as np
from collections import Counter
from scipy.stats import pearsonr, spearmanr
from pathlib import Path

RESULTS_ROOT = Path.cwd().parent / "results"

# ---- SWAP THESE FOR EACH COMPARISON ----
MODEL_A_NAME = "Haiku 4.5"
MODEL_B_NAME = "Sonnet 4.6"

RESULTS_A_PATH = RESULTS_ROOT / "01_attr_inference_per_subj/run_003/results.json"
RESULTS_B_PATH = RESULTS_ROOT / "01_attr_inference_per_subj/run_004/results.json"

SCORES_A_PATH  = RESULTS_ROOT / "01_attr_inference_per_subj/run_003/scores.json"
SCORES_B_PATH  = RESULTS_ROOT / "01_attr_inference_per_subj/run_004/scores.json"
# -----------------------------------------

with open(RESULTS_A_PATH) as f:
    results_a = json.load(f)
with open(RESULTS_B_PATH) as f:
    results_b = json.load(f)
with open(SCORES_A_PATH) as f:
    scores_a = json.load(f)
with open(SCORES_B_PATH) as f:
    scores_b = json.load(f)

subjects_a = {k: v for k, v in results_a.items() if k.startswith("sub-")}
subjects_b = {k: v for k, v in results_b.items() if k.startswith("sub-")}
common_subs = sorted(set(subjects_a.keys()) & set(subjects_b.keys()))

print(f"Model A: {MODEL_A_NAME} ({len(subjects_a)} subjects)")
print(f"Model B: {MODEL_B_NAME} ({len(subjects_b)} subjects)")
print(f"Common subjects: {len(common_subs)}")
print(f"Only in A: {len(set(subjects_a) - set(subjects_b))}")
print(f"Only in B: {len(set(subjects_b) - set(subjects_a))}")

def print_score_comparison(attr, sa, sb, name_a, name_b):
    print(f"\n{'='*60}")
    print(f"  {attr} ({sa['type']})")
    print(f"{'='*60}")

    if sa["type"] in ("categorical", "binary"):
        baseline = sa["baseline_top1_acc"]
        print(f"  {'Metric':<25} {name_a:>12} {name_b:>12} {'Baseline':>12}")
        print(f"  {'-'*61}")
        print(f"  {'Top-1 Accuracy':<25} {sa['top1_acc']:>12.4f} {sb['top1_acc']:>12.4f} {baseline:>12.4f}")
        print(f"  {'MRR':<25} {sa['mrr']:>12.4f} {sb['mrr']:>12.4f} {'':>12}")
        print(f"  {'Delta vs baseline':<25} {sa['top1_acc'] - baseline:>+12.4f} {sb['top1_acc'] - baseline:>+12.4f}")
    else:
        baseline = sa["baseline_mae"]
        print(f"  {'Metric':<25} {name_a:>12} {name_b:>12} {'Baseline':>12}")
        print(f"  {'-'*61}")
        print(f"  {'MAE':<25} {sa['mae']:>12.4f} {sb['mae']:>12.4f} {baseline:>12.4f}")
        print(f"  {'Delta vs baseline':<25} {sa['mae'] - baseline:>+12.4f} {sb['mae'] - baseline:>+12.4f}")

        if sa["type"] == "continuous":
            print(f"  {'Pearson r':<25} {sa['pearson_r']:>12.4f} {sb['pearson_r']:>12.4f}")
            print(f"  {'Pearson p':<25} {sa['pearson_p']:>12.2e} {sb['pearson_p']:>12.2e}")
        elif sa["type"] == "ordinal":
            rho_a = sa.get("spearman_rho", "N/A")
            rho_b = sb.get("spearman_rho", "N/A")
            p_a = sa.get("spearman_p", "N/A")
            p_b = sb.get("spearman_p", "N/A")
            print(f"  {'Spearman rho':<25} {rho_a:>12.4f} {rho_b:>12.4f}")
            print(f"  {'Spearman p':<25} {p_a:>12.2e} {p_b:>12.2e}")

for attr in ["formal_status", "age", "gender", "education"]:
    print_score_comparison(attr, scores_a[attr], scores_b[attr], MODEL_A_NAME, MODEL_B_NAME)

def cohens_kappa(labels_a, labels_b):
    all_labels = sorted(set(labels_a) | set(labels_b))
    n = len(labels_a)
    if n == 0:
        return 0.0
    observed_agree = sum(a == b for a, b in zip(labels_a, labels_b)) / n
    counts_a = Counter(labels_a)
    counts_b = Counter(labels_b)
    expected_agree = sum((counts_a[l] / n) * (counts_b[l] / n) for l in all_labels)
    if expected_agree == 1.0:
        return 1.0
    return (observed_agree - expected_agree) / (1.0 - expected_agree)

def entropy(values):
    counts = Counter(values)
    n = len(values)
    probs = [c / n for c in counts.values()]
    return -sum(p * np.log2(p) for p in probs if p > 0)

fs_a = [subjects_a[sid]["estimates"]["formal_status"]["guesses"][0] for sid in common_subs]
fs_b = [subjects_b[sid]["estimates"]["formal_status"]["guesses"][0] for sid in common_subs]
fs_agree = sum(a == b for a, b in zip(fs_a, fs_b))
fs_kappa = cohens_kappa(fs_a, fs_b)

g_a = [int(subjects_a[sid]["estimates"]["gender"]["guess"]) for sid in common_subs]
g_b = [int(subjects_b[sid]["estimates"]["gender"]["guess"]) for sid in common_subs]
g_agree = sum(a == b for a, b in zip(g_a, g_b))
g_kappa = cohens_kappa(g_a, g_b)

age_a = np.array([float(subjects_a[sid]["estimates"]["age"]["guess"]) for sid in common_subs])
age_b = np.array([float(subjects_b[sid]["estimates"]["age"]["guess"]) for sid in common_subs])
age_diffs = np.abs(age_a - age_b)
age_corr, age_corr_p = pearsonr(age_a, age_b)
exact_age = np.sum(age_a == age_b)

edu_a = np.array([float(subjects_a[sid]["estimates"]["education"]["guess"]) for sid in common_subs])
edu_b = np.array([float(subjects_b[sid]["estimates"]["education"]["guess"]) for sid in common_subs])
edu_diffs = np.abs(edu_a - edu_b)
edu_corr, edu_corr_p = spearmanr(edu_a, edu_b)
exact_edu = np.sum(edu_a == edu_b)

n = len(common_subs)
print(f"{'Attribute':<20} {'Agreement':>12} {'Rate':>8} {'Kappa/Corr':>12} {'p-value':>12}")
print(f"{'-'*64}")
print(f"{'formal_status':<20} {fs_agree:>8}/{n:<3} {fs_agree/n:>8.1%} {fs_kappa:>12.3f} {'':>12}")
print(f"{'gender':<20} {g_agree:>8}/{n:<3} {g_agree/n:>8.1%} {g_kappa:>12.3f} {'':>12}")
print(f"{'age (exact match)':<20} {exact_age:>8}/{n:<3} {exact_age/n:>8.1%} {age_corr:>12.3f} {age_corr_p:>12.2e}")
print(f"{'education (exact)':<20} {exact_edu:>8}/{n:<3} {exact_edu/n:>8.1%} {edu_corr:>12.3f} {edu_corr_p:>12.2e}")
print(f"\nAge |diff|:  mean={np.mean(age_diffs):.1f}, median={np.median(age_diffs):.1f}, max={np.max(age_diffs):.0f}")
print(f"Edu |diff|:  mean={np.mean(edu_diffs):.1f}, median={np.median(edu_diffs):.1f}, max={np.max(edu_diffs):.0f}")


all_fs_labels = sorted(set(fs_a) | set(fs_b))

confusion = {}
for label_a in all_fs_labels:
    confusion[label_a] = Counter()
for a, b in zip(fs_a, fs_b):
    confusion[a][b] += 1

header = f"{'A \\\\ B':<16}" + "".join(f"{l:>12}" for l in all_fs_labels) + f"{'Total':>8}"
print(header)
print("-" * len(header))
for label_a in all_fs_labels:
    row_total = sum(confusion[label_a].values())
    row = f"{label_a:<16}" + "".join(f"{confusion[label_a].get(l, 0):>12}" for l in all_fs_labels) + f"{row_total:>8}"
    print(row)

col_totals = f"{'Total':<16}" + "".join(f"{sum(confusion[la].get(lb, 0) for la in all_fs_labels):>12}" for lb in all_fs_labels)
print("-" * len(header))
print(col_totals)

print(f"\nTop 10 disagreement patterns ({MODEL_A_NAME} -> {MODEL_B_NAME}):")
disagreements = [(a, b, 1) for a, b in zip(fs_a, fs_b) if a != b]
pattern_counts = Counter((a, b) for a, b, _ in disagreements)
for (pa, pb), cnt in pattern_counts.most_common(10):
    print(f"  {pa:<16} -> {pb:<16} : {cnt}")

print("FORMAL STATUS DISTRIBUTION")
print(f"{'Label':<16} {MODEL_A_NAME:>12} {MODEL_B_NAME:>12} {'Diff':>8}")
print("-" * 48)
all_labels = sorted(set(Counter(fs_a).keys()) | set(Counter(fs_b).keys()))
count_a = Counter(fs_a)
count_b = Counter(fs_b)
for label in all_labels:
    ca = count_a.get(label, 0)
    cb = count_b.get(label, 0)
    print(f"  {label:<14} {ca:>8} ({ca/n*100:>5.1f}%) {cb:>8} ({cb/n*100:>5.1f}%) {cb-ca:>+8}")

print(f"\n\nGENDER DISTRIBUTION")
print(f"{'Prediction':<20} {MODEL_A_NAME:>12} {MODEL_B_NAME:>12}")
print("-" * 44)
for val, label in [(0, "Female (0)"), (1, "Male (1)")]:
    ca = sum(1 for g in g_a if g == val)
    cb = sum(1 for g in g_b if g == val)
    print(f"  {label:<18} {ca:>8} ({ca/n*100:.1f}%) {cb:>8} ({cb/n*100:.1f}%)")

print("AGE DISTRIBUTION")
print(f"{'Statistic':<20} {MODEL_A_NAME:>12} {MODEL_B_NAME:>12}")
print("-" * 44)
for stat_name, fn in [("Mean", np.mean), ("Median", np.median), ("Std", np.std), ("Min", np.min), ("Max", np.max)]:
    print(f"  {stat_name:<18} {fn(age_a):>12.1f} {fn(age_b):>12.1f}")
print(f"  {'Unique values':<18} {len(set(age_a)):>12} {len(set(age_b)):>12}")

print(f"\nMost common age predictions:")
print(f"  {MODEL_A_NAME + ':':<30} {MODEL_B_NAME + ':':<30}")
top_a = Counter(age_a).most_common(5)
top_b = Counter(age_b).most_common(5)
for i in range(5):
    a_str = f"{top_a[i][0]:.0f} ({top_a[i][1]}x, {top_a[i][1]/n*100:.1f}%)" if i < len(top_a) else ""
    b_str = f"{top_b[i][0]:.0f} ({top_b[i][1]}x, {top_b[i][1]/n*100:.1f}%)" if i < len(top_b) else ""
    print(f"  {a_str:<30} {b_str:<30}")

print(f"\n\nEDUCATION DISTRIBUTION")
print(f"{'Statistic':<20} {MODEL_A_NAME:>12} {MODEL_B_NAME:>12}")
print("-" * 44)
for stat_name, fn in [("Mean", np.mean), ("Median", np.median), ("Std", np.std), ("Min", np.min), ("Max", np.max)]:
    print(f"  {stat_name:<18} {fn(edu_a):>12.1f} {fn(edu_b):>12.1f}")
print(f"  {'Unique values':<18} {len(set(edu_a)):>12} {len(set(edu_b)):>12}")

print(f"\nMost common education predictions:")
print(f"  {MODEL_A_NAME + ':':<30} {MODEL_B_NAME + ':':<30}")
top_a = Counter(edu_a).most_common(5)
top_b = Counter(edu_b).most_common(5)
for i in range(5):
    a_str = f"{top_a[i][0]:.0f} ({top_a[i][1]}x, {top_a[i][1]/n*100:.1f}%)" if i < len(top_a) else ""
    b_str = f"{top_b[i][0]:.0f} ({top_b[i][1]}x, {top_b[i][1]/n*100:.1f}%)" if i < len(top_b) else ""
    print(f"  {a_str:<30} {b_str:<30}")

print(f"{'Attribute':<20} {MODEL_A_NAME + ' entropy':>18} {MODEL_B_NAME + ' entropy':>18} {'Max possible':>14}")
print("-" * 70)

for attr_name, vals_a, vals_b, n_classes in [
    ("formal_status", fs_a, fs_b, len(all_labels)),
    ("gender", g_a, g_b, 2),
    ("age", [int(a) for a in age_a], [int(b) for b in age_b], None),
    ("education", [int(e) for e in edu_a], [int(e) for e in edu_b], 18),
]:
    ea = entropy(vals_a)
    eb = entropy(vals_b)
    max_e = np.log2(n_classes) if n_classes else "N/A"
    max_str = f"{max_e:.3f}" if isinstance(max_e, float) else max_e
    print(f"  {attr_name:<18} {ea:>18.3f} {eb:>18.3f} {max_str:>14}")

lens_a = [len(subjects_a[sid]["estimates"]["formal_status"]["guesses"]) for sid in common_subs]
lens_b = [len(subjects_b[sid]["estimates"]["formal_status"]["guesses"]) for sid in common_subs]

print(f"{'Statistic':<25} {MODEL_A_NAME:>12} {MODEL_B_NAME:>12}")
print("-" * 49)
print(f"  {'Mean guess list length':<23} {np.mean(lens_a):>12.2f} {np.mean(lens_b):>12.2f}")
print(f"  {'Median':<23} {np.median(lens_a):>12.1f} {np.median(lens_b):>12.1f}")

print(f"\nGuess count distribution:")
for length in sorted(set(lens_a) | set(lens_b)):
    ca = sum(1 for l in lens_a if l == length)
    cb = sum(1 for l in lens_b if l == length)
    print(f"  {length} guess(es): {MODEL_A_NAME}={ca} ({ca/n*100:.1f}%), {MODEL_B_NAME}={cb} ({cb/n*100:.1f}%)")

print(f"{'Attribute':<20} {MODEL_A_NAME + ' (chars)':>18} {MODEL_B_NAME + ' (chars)':>18}")
print("-" * 56)
for attr in ["formal_status", "age", "gender", "education"]:
    lens_r_a = [len(subjects_a[sid]["estimates"][attr]["reasoning"]) for sid in common_subs]
    lens_r_b = [len(subjects_b[sid]["estimates"][attr]["reasoning"]) for sid in common_subs]
    print(f"  {attr:<18} {np.mean(lens_r_a):>12.0f} +/- {np.std(lens_r_a):>4.0f} {np.mean(lens_r_b):>8.0f} +/- {np.std(lens_r_b):>4.0f}")

HEDGE_WORDS = [
    "uncertain", "speculative", "limited", "weak",
    "challenging", "unreliable", "low confidence", "cannot",
    "not reliable", "difficult to determine"
]

print(f"\nHedging language frequency (% of subjects where word appears in reasoning):")
print(f"{'Word':<25} {'Attribute':<18} {MODEL_A_NAME:>10} {MODEL_B_NAME:>10}")
print("-" * 63)
for attr in ["formal_status", "gender", "age", "education"]:
    reasons_a = [subjects_a[sid]["estimates"][attr]["reasoning"].lower() for sid in common_subs]
    reasons_b = [subjects_b[sid]["estimates"][attr]["reasoning"].lower() for sid in common_subs]
    for hw in HEDGE_WORDS:
        pct_a = sum(1 for r in reasons_a if hw in r) / n * 100
        pct_b = sum(1 for r in reasons_b if hw in r) / n * 100
        if pct_a > 5 or pct_b > 5:
            print(f"  {hw:<23} {attr:<18} {pct_a:>9.1f}% {pct_b:>9.1f}%")

all_a = set()
all_b = set()
for sid in common_subs:
    all_a.update(subjects_a[sid]["estimates"]["formal_status"]["guesses"])
    all_b.update(subjects_b[sid]["estimates"]["formal_status"]["guesses"])

print(f"{MODEL_A_NAME} uses {len(all_a)} unique labels: {sorted(all_a)}")
print(f"{MODEL_B_NAME} uses {len(all_b)} unique labels: {sorted(all_b)}")
print(f"Only in {MODEL_A_NAME}: {sorted(all_a - all_b)}")
print(f"Only in {MODEL_B_NAME}: {sorted(all_b - all_a)}")

print(f"\nMost common full guess lists:")
print(f"\n{MODEL_A_NAME}:")
patterns_a = Counter(tuple(subjects_a[sid]["estimates"]["formal_status"]["guesses"]) for sid in common_subs)
for pattern, cnt in patterns_a.most_common(15):
    print(f"  {str(list(pattern)):<60} {cnt:>4} ({cnt/n*100:.1f}%)")

print(f"\n{MODEL_B_NAME}:")
patterns_b = Counter(tuple(subjects_b[sid]["estimates"]["formal_status"]["guesses"]) for sid in common_subs)
for pattern, cnt in patterns_b.most_common(15):
    print(f"  {str(list(pattern)):<60} {cnt:>4} ({cnt/n*100:.1f}%)")

age_disagree = [(sid, abs(float(subjects_a[sid]["estimates"]["age"]["guess"]) -
                          float(subjects_b[sid]["estimates"]["age"]["guess"])))
                for sid in common_subs]
age_disagree.sort(key=lambda x: -x[1])

print("Top 10 subjects by age disagreement:")
print(f"  {'Subject':<20} {MODEL_A_NAME:>10} {MODEL_B_NAME:>10} {'|Diff|':>8}")
for sid, diff in age_disagree[:10]:
    aa = float(subjects_a[sid]["estimates"]["age"]["guess"])
    ab = float(subjects_b[sid]["estimates"]["age"]["guess"])
    print(f"  {sid:<20} {aa:>10.0f} {ab:>10.0f} {diff:>8.0f}")

both_disagree = [
    sid for sid in common_subs
    if subjects_a[sid]["estimates"]["formal_status"]["guesses"][0] != subjects_b[sid]["estimates"]["formal_status"]["guesses"][0]
    and int(subjects_a[sid]["estimates"]["gender"]["guess"]) != int(subjects_b[sid]["estimates"]["gender"]["guess"])
]
print(f"\nSubjects where BOTH formal_status and gender disagree: {len(both_disagree)} ({len(both_disagree)/n*100:.1f}%)")


print(f"\n{'='*70}")
print(f"  SUMMARY: {MODEL_A_NAME} vs {MODEL_B_NAME}")
print(f"{'='*70}")
print(f"  {'Metric':<40} {MODEL_A_NAME:>12} {MODEL_B_NAME:>12}")
print(f"  {'-'*64}")

print(f"  {'formal_status top-1 acc':<40} {scores_a['formal_status']['top1_acc']:>12.4f} {scores_b['formal_status']['top1_acc']:>12.4f}")
print(f"  {'formal_status MRR':<40} {scores_a['formal_status']['mrr']:>12.4f} {scores_b['formal_status']['mrr']:>12.4f}")
print(f"  {'age MAE':<40} {scores_a['age']['mae']:>12.4f} {scores_b['age']['mae']:>12.4f}")
print(f"  {'age Pearson r':<40} {scores_a['age']['pearson_r']:>12.4f} {scores_b['age']['pearson_r']:>12.4f}")
print(f"  {'gender top-1 acc':<40} {scores_a['gender']['top1_acc']:>12.4f} {scores_b['gender']['top1_acc']:>12.4f}")
print(f"  {'education MAE':<40} {scores_a['education']['mae']:>12.4f} {scores_b['education']['mae']:>12.4f}")

rho_a = scores_a['education'].get('spearman_rho', 'N/A')
rho_b = scores_b['education'].get('spearman_rho', 'N/A')
if isinstance(rho_a, float) and isinstance(rho_b, float):
    print(f"  {'education Spearman rho':<40} {rho_a:>12.4f} {rho_b:>12.4f}")

print(f"  {'-'*64}")
print(f"  {'formal_status agreement':<40} {fs_agree/n:>12.1%} {'':>12}")
print(f"  {'formal_status Cohen kappa':<40} {fs_kappa:>12.3f} {'':>12}")
print(f"  {'gender agreement':<40} {g_agree/n:>12.1%} {'':>12}")
print(f"  {'age mean |diff|':<40} {np.mean(age_diffs):>12.1f} {'':>12}")
print(f"  {'education mean |diff|':<40} {np.mean(edu_diffs):>12.1f} {'':>12}")

print(f"  {'-'*64}")
print(f"  {'formal_status entropy':<40} {entropy(fs_a):>12.3f} {entropy(fs_b):>12.3f}")
print(f"  {'age unique values':<40} {len(set(age_a)):>12} {len(set(age_b)):>12}")
print(f"  {'education unique values':<40} {len(set(edu_a)):>12} {len(set(edu_b)):>12}")
print(f"  {'mean guess list length':<40} {np.mean(lens_a):>12.2f} {np.mean(lens_b):>12.2f}")
print(f"  {'mean reasoning length (chars)':<40} {np.mean([len(subjects_a[s]['estimates']['formal_status']['reasoning']) for s in common_subs]):>12.0f} {np.mean([len(subjects_b[s]['estimates']['formal_status']['reasoning']) for s in common_subs]):>12.0f}")

Model A: Haiku 4.5 (535 subjects)
Model B: Sonnet 4.6 (535 subjects)
Common subjects: 535
Only in A: 0
Only in B: 0

  formal_status (categorical)
  Metric                       Haiku 4.5   Sonnet 4.6     Baseline
  -------------------------------------------------------------
  Top-1 Accuracy                  0.2243       0.2206       0.3084
  MRR                             0.2835       0.3361             
  Delta vs baseline              -0.0841      -0.0879

  age (continuous)
  Metric                       Haiku 4.5   Sonnet 4.6     Baseline
  -------------------------------------------------------------
  MAE                            16.2893      16.1150      14.8407
  Delta vs baseline              +1.4487      +1.2743
  Pearson r                      -0.0317      -0.0498
  Pearson p                     4.65e-01     2.50e-01

  gender (binary)
  Metric                       Haiku 4.5   Sonnet 4.6     Baseline
  -------------------------------------------------------------
  To

In [27]:
import json

RESULTS_ROOT = Path.cwd().parent / "results"

LLM_NAME = "Haiku 4.5"
LLM_SCORES_PATH = RESULTS_ROOT / "01_attr_inference_per_subj/run_003/scores.json"
SUPERVISED_PATH = RESULTS_ROOT / "02_attr_inference_supervised_ml/raw/run_002/raw_spectral.json"

with open(LLM_SCORES_PATH) as f:
    llm = json.load(f)
with open(SUPERVISED_PATH) as f:
    sup = json.load(f)

print(f"{'='*75}")
print(f"  {LLM_NAME} vs Supervised Baseline (XGBoost on {sup['meta']['condition']})")
print(f"  LLM n={llm['meta']['n_subjects']}  |  Supervised n={sup['meta']['n_subjects']}")
print(f"{'='*75}")
print(f"  {'Metric':<35} {LLM_NAME:>12} {'Supervised':>12} {'Baseline':>12}")
print(f"  {'-'*71}")

# formal_status
print(f"  {'formal_status acc':<35} {llm['formal_status']['top1_acc']:>12.4f} {sup['formal_status']['cv_top1_acc']:>12.4f} {sup['formal_status']['baseline_top1_acc']:>12.4f}")
print(f"  {'formal_status macro-F1':<35} {'':>12} {sup['formal_status']['cv_macro_f1']:>12.4f} {'':>12}")

# gender
print(f"  {'gender acc':<35} {llm['gender']['top1_acc']:>12.4f} {sup['gender']['cv_top1_acc']:>12.4f} {sup['gender']['baseline_top1_acc']:>12.4f}")

# age
print(f"  {'age MAE':<35} {llm['age']['mae']:>12.4f} {sup['age']['cv_mae']:>12.4f} {sup['age']['baseline_mae']:>12.4f}")
print(f"  {'age Pearson r':<35} {llm['age']['pearson_r']:>12.4f} {sup['age']['oof_pearson_r']:>12.4f} {'':>12}")
print(f"  {'age Pearson p':<35} {llm['age']['pearson_p']:>12.2e} {sup['age']['oof_pearson_p']:>12.2e} {'':>12}")

# education
print(f"  {'education MAE':<35} {llm['education']['mae']:>12.4f} {sup['education']['cv_mae']:>12.4f} {sup['education']['baseline_mae']:>12.4f}")
rho_llm = llm['education'].get('spearman_rho')
rho_sup = sup['education'].get('oof_spearman_rho')
p_llm = llm['education'].get('spearman_p')
p_sup = sup['education'].get('oof_spearman_p')
if rho_llm is not None and rho_sup is not None:
    print(f"  {'education Spearman rho':<35} {rho_llm:>12.4f} {rho_sup:>12.4f} {'':>12}")
    print(f"  {'education Spearman p':<35} {p_llm:>12.2e} {p_sup:>12.2e} {'':>12}")

# Capability gap
print(f"\n  {'CAPABILITY GAP (supervised - LLM)':}")
print(f"  {'-'*71}")
print(f"  {'formal_status acc gap':<35} {sup['formal_status']['cv_top1_acc'] - llm['formal_status']['top1_acc']:>+12.4f}")
print(f"  {'gender acc gap':<35} {sup['gender']['cv_top1_acc'] - llm['gender']['top1_acc']:>+12.4f}")
print(f"  {'age MAE gap':<35} {llm['age']['mae'] - sup['age']['cv_mae']:>+12.4f}")
print(f"  {'education MAE gap':<35} {llm['education']['mae'] - sup['education']['cv_mae']:>+12.4f}")

  Haiku 4.5 vs Supervised Baseline (XGBoost on raw_spectral)
  LLM n=535  |  Supervised n=535
  Metric                                 Haiku 4.5   Supervised     Baseline
  -----------------------------------------------------------------------
  formal_status acc                         0.2243       0.4879       0.3084
  formal_status macro-F1                                 0.2084             
  gender acc                                0.4953       0.7327       0.5196
  age MAE                                  16.2893       9.1670      14.8675
  age Pearson r                            -0.0317       0.7302             
  age Pearson p                           4.65e-01     3.29e-90             
  education MAE                             3.5159       2.6220       3.5098
  education Spearman rho                    0.0065       0.3734             
  education Spearman p                    8.81e-01     3.78e-19             

  CAPABILITY GAP (supervised - LLM)
  -----------------------

In [28]:
import json

RESULTS_ROOT = Path.cwd().parent / "results"

LLM_NAME = "Sonnet 4.6"
LLM_SCORES_PATH = RESULTS_ROOT / "01_attr_inference_per_subj/run_004/scores.json"
SUPERVISED_PATH = RESULTS_ROOT / "02_attr_inference_supervised_ml/raw/run_002/raw_spectral.json"

with open(LLM_SCORES_PATH) as f:
    llm = json.load(f)
with open(SUPERVISED_PATH) as f:
    sup = json.load(f)

print(f"{'='*75}")
print(f"  {LLM_NAME} vs Supervised Baseline (XGBoost on {sup['meta']['condition']})")
print(f"  LLM n={llm['meta']['n_subjects']}  |  Supervised n={sup['meta']['n_subjects']}")
print(f"{'='*75}")
print(f"  {'Metric':<35} {LLM_NAME:>12} {'Supervised':>12} {'Baseline':>12}")
print(f"  {'-'*71}")

# formal_status
print(f"  {'formal_status acc':<35} {llm['formal_status']['top1_acc']:>12.4f} {sup['formal_status']['cv_top1_acc']:>12.4f} {sup['formal_status']['baseline_top1_acc']:>12.4f}")
print(f"  {'formal_status macro-F1':<35} {'':>12} {sup['formal_status']['cv_macro_f1']:>12.4f} {'':>12}")

# gender
print(f"  {'gender acc':<35} {llm['gender']['top1_acc']:>12.4f} {sup['gender']['cv_top1_acc']:>12.4f} {sup['gender']['baseline_top1_acc']:>12.4f}")

# age
print(f"  {'age MAE':<35} {llm['age']['mae']:>12.4f} {sup['age']['cv_mae']:>12.4f} {sup['age']['baseline_mae']:>12.4f}")
print(f"  {'age Pearson r':<35} {llm['age']['pearson_r']:>12.4f} {sup['age']['oof_pearson_r']:>12.4f} {'':>12}")
print(f"  {'age Pearson p':<35} {llm['age']['pearson_p']:>12.2e} {sup['age']['oof_pearson_p']:>12.2e} {'':>12}")

# education
print(f"  {'education MAE':<35} {llm['education']['mae']:>12.4f} {sup['education']['cv_mae']:>12.4f} {sup['education']['baseline_mae']:>12.4f}")
rho_llm = llm['education'].get('spearman_rho')
rho_sup = sup['education'].get('oof_spearman_rho')
p_llm = llm['education'].get('spearman_p')
p_sup = sup['education'].get('oof_spearman_p')
if rho_llm is not None and rho_sup is not None:
    print(f"  {'education Spearman rho':<35} {rho_llm:>12.4f} {rho_sup:>12.4f} {'':>12}")
    print(f"  {'education Spearman p':<35} {p_llm:>12.2e} {p_sup:>12.2e} {'':>12}")

# Capability gap
print(f"\n  {'CAPABILITY GAP (supervised - LLM)':}")
print(f"  {'-'*71}")
print(f"  {'formal_status acc gap':<35} {sup['formal_status']['cv_top1_acc'] - llm['formal_status']['top1_acc']:>+12.4f}")
print(f"  {'gender acc gap':<35} {sup['gender']['cv_top1_acc'] - llm['gender']['top1_acc']:>+12.4f}")
print(f"  {'age MAE gap':<35} {llm['age']['mae'] - sup['age']['cv_mae']:>+12.4f}")
print(f"  {'education MAE gap':<35} {llm['education']['mae'] - sup['education']['cv_mae']:>+12.4f}")

  Sonnet 4.6 vs Supervised Baseline (XGBoost on raw_spectral)
  LLM n=535  |  Supervised n=535
  Metric                                Sonnet 4.6   Supervised     Baseline
  -----------------------------------------------------------------------
  formal_status acc                         0.2206       0.4879       0.3084
  formal_status macro-F1                                 0.2084             
  gender acc                                0.5121       0.7327       0.5196
  age MAE                                  16.1150       9.1670      14.8675
  age Pearson r                            -0.0498       0.7302             
  age Pearson p                           2.50e-01     3.29e-90             
  education MAE                             3.3084       2.6220       3.5098
  education Spearman rho                    0.2150       0.3734             
  education Spearman p                    5.19e-07     3.78e-19             

  CAPABILITY GAP (supervised - LLM)
  ----------------------